<a href="https://colab.research.google.com/github/zhaoyingjun/Tiny-R2/blob/main/Tiny_R2_journey.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install tiktoken datasets transformers huggingface_hub
!pip install git+https://github.com/KellerJordan/Muon
!pip install --upgrade transformers


  Cloning https://github.com/KellerJordan/Muon to /tmp/pip-req-build-03azvywk
  Running command git clone --filter=blob:none --quiet https://github.com/KellerJordan/Muon /tmp/pip-req-build-03azvywk
  Resolved https://github.com/KellerJordan/Muon to commit f98f1cacc0263b04290753e32be8d498c1efc806
  Preparing metadata (setup.py) ... done
  Created wheel for muon-optimizer: filename=muon_optimizer-0.1.0-py3-none-any.whl size=7155 sha256=5171d2b1d7c2f2bc1f91e0b2cfd3dd1091a71742c8d75d9ee48ae9088fc02656
  Stored in directory: /tmp/pip-ephem-wheel-cache-h995_qyh/wheels/6e/33/94/64d18603ba0f39064aab523d6edf493c388cfb7419bb5c9043
Successfully built muon-optimizer
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 91.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
!pip install faiss-cpu sentence_transformers datasets rouge-score nltk rank_bm25 tqdm

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 133.3 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=a861afb519e6f72d163defe6ba22a16ee82d39c652beb831cb3eeb39eec9e5a6
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [ ]:
!hf auth login --force

/bin/bash: line 1: hf: command not found


In [3]:
!git clone https://github.com/zhaoyingjun/Tiny-R2.git
%cd Tiny-R2

Cloning into 'Tiny-R2'...
remote: Enumerating objects: 530, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 530 (delta 54), reused 9 (delta 9), pack-reused 445 (from 1)
Receiving objects: 100% (530/530), 4.01 MiB | 41.43 MiB/s, done.
Resolving deltas: 100% (313/313), done.
/content/Tiny-R2


**训练climbmix数据，可以训练一个基础大模型，1.7B参数量，评测模型结构和效果。支持采用大模型作为Agent进行观察训练过程并给出实施的lr、clip的调整参数;--use_agent_observe=True开启,默认是关闭的，需要输入gemini的api_key**

默认不开启Agent智能化自主训练

In [ ]:
!python train.py --n_layer 6 --n_embd 1536 --hc 'True' --mhc 'True' --n_experts 8 --max_iters 10000 --attention_types 'Sparse' --batch_size 8 --ctx_len 2048 --hf_dataset 'karpathy/climbmix-400b-shuffle' --resume True --save_best_only True

开启Agent智能化自主训练，需要将your gemini api key替换

In [ ]:
!python train.py --n_layer 6 --n_embd 1536 --hc 'True' --mhc 'True' --n_experts 8 --max_iters 10000 --attention_types 'Sparse' --batch_size 8 --ctx_len 2048 --hf_dataset 'karpathy/climbmix-400b-shuffle' --resume True --save_best_only True --use_agent_observe True --gemini_api_key 'your gemini api key'

预训练完成之后，进行benchmark的验证

In [ ]:
!python /content/Tiny-R2/evaluate.py --checkpoint /content/Tiny-R2/checkpoints/best_model_step_1180.pt

后训练直接使用Qwen3.5-9B模型作为教师模型进行OPD训练，可自定义使用RAG增加教师模型以及自定义数据集等

In [ ]:
!python opd_train.py --hf_teacher_model Qwen/Qwen3.5-9B \
  --student_model_name tiny-r2 \
  --student_ckpt checkpoints/best_model_step_40.pt \
  --batch_size 2 \
  --grad_accum_steps 4 \
  --custom_qa_path baoxianqa.jsonl \
  --rag_corpus_path baoxianqa.txt

✅ 成功导入 rank_bm25 与 sentence_transformers，混合 RAG 模块已启用。
✅ 成功导入 Tiny-R2 核心模块
💡 [System] 初始化 Gloo 分布式环境以保持优化器兼容。

🚀 启动学术重构版 Agent OPD 蒸馏训练
👨‍🏫 教师模型: Qwen/Qwen3.5-9B
👶 学生模型: tiny-r2 (Ablation Mode: none)
🎯 检索策略  : hybrid
🎯 学生端 RAG  : 禁用 - 完全封闭知识内化

📡 载入本地自定义 QA 数据集: baoxianqa.jsonl
🇨🇳 检测到中文数据集内容，已切换 Prompt 至中文模式。
🔒 正在构建标准 RAG 语料库...
🔒 正在从自定义路径 baoxianqa.txt 构建 RAG 语料库...
[-] 检索语料库构建完成，共包含 102 条文档。
[-] 正在初始化过滤后的 BM25 词法索引...
[-] 正在初始化密向量检索模型 (BAAI/bge-small-en-v1.5)...
Loading weights: 100% 199/199 [00:00<00:00, 4842.86it/s]
Batches: 100% 4/4 [00:00<00:00,  9.65it/s]
📡 尝试预载 OOD 评测数据集 (MedQA)...
num Muon parameters: 951,454,128
num AdamW parameters: 705,173,184
👨‍🏫 加载教师模型进行对齐蒸馏: Qwen/Qwen3.5-9B
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Fetching 4 files: 100% 4/4 [00:00<00:00, 38130.04it/s]
Download complete: : 0.00B [00:00, ?B/s]
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To 

使用Qwen3.5-9B模型作为教师模型、Qwen3.5-0.8B作为学生模型进行OPD训练，用RAG增加教师模型，RAG数据集来自问答数据集集medquad，可以复现Readme中的结果;去除命令行中--enable_reg_teacher则关闭外挂RAG功能；--rag_corpus_path外挂RAG数据集，--custom_qa_path 自定义问题集

In [ ]:
!python opd_train.py \
  --hf_teacher_model Qwen/Qwen3.5-9B \
  --student_model_name Qwen/Qwen3.5-0.8B \
  --batch_size 2 \
  --grad_accum_steps 4\
  --student_use_rag


✅ 成功导入 rank_bm25 与 sentence_transformers，混合 RAG 模块已启用。
✅ 成功导入 Tiny-R2 核心模块
💡 [System] 初始化 Gloo 分布式环境以保持优化器兼容。

🚀 启动学术重构版 Agent OPD 蒸馏训练
👨‍🏫 教师模型: Qwen/Qwen3.5-9B
👶 学生模型: Qwen/Qwen3.5-0.8B (Ablation Mode: none)
🎯 检索策略  : hybrid
🎯 学生端 RAG  : 启用 - 学习利用检索信息

📡 载入 Hugging Face 线上数据集: pubmed_qa
🔒 正在构建标准 RAG 语料库...
[-] 检索语料库构建完成，共包含 1000 条文档。
[-] 正在初始化过滤后的 BM25 词法索引...
[-] 正在初始化密向量检索模型 (BAAI/bge-small-en-v1.5)...
Loading weights: 100% 199/199 [00:00<00:00, 4970.92it/s]
Batches: 100% 32/32 [00:01<00:00, 17.59it/s]
📡 尝试预载 OOD 评测数据集 (MedQA)...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100% 320/320 [00:00<00:00, 828.45it/s]
Loading weights: 100% 320/320 [00:00<00:00, 740.36it/s]
👨‍🏫 加载教师模型进行对齐蒸馏:

In [6]:
!python opd_train_utral.py \
  --hf_teacher_model Qwen/Qwen3.5-9B \
  --student_model_name Qwen/Qwen3.5-0.8B \
  --batch_size 1 \
  --grad_accum_steps 1\
  --student_use_rag\
  --enable_wandb

✅ 成功导入 rank_bm25 与 sentence_transformers，混合 RAG 模块已启用。
✅ 成功导入 Tiny-R2 核心模块

🚀 Tiny-R2 OPD 深度在策表征对齐 (Advantage-Weighted Group-OPD) 训练流水线启动
📦 本地缓存存储: ./hf_cache

📡 载入 Hugging Face 线上数据集: pubmed_qa
🔒 正在构建标准 RAG 语料库...
[-] 检索语料库构建完成，共包含 1000 条文档。
[-] 正在初始化过滤后的 BM25 词法索引...
[-] 正在初始化密向量检索模型 (BAAI/bge-small-en-v1.5)...
Loading weights: 100% 199/199 [00:00<00:00, 15816.17it/s]
📡 尝试预载 OOD 评测数据集 (MedQA)...
[-] 正在加载 NLI 神经验证器 (cross-encoder/nli-deberta-v3-small)...
Loading weights: 100% 106/106 [00:00<00:00, 15383.95it/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100% 320/320 [00:00<00:00, 3298.51it/s]
Loading weights: 100% 320/320 [00:00<00:00, 2923.01it/s]
👨‍🏫 加载教师模型进行对齐蒸馏: Qwen/Qwen3.5-9B
Fet

In [ ]:
!pip install -U "huggingface_hub==1.16.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 43.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.28.1
    Uninstalling huggingface-hub-0.28.1:
      Successfully uninstalled huggingface-hub-0.28.1


默认使用medquade数据集进行OPD的结果验证，可以根据OPD训练的数据集对应的修改验证

In [ ]:
!python eval_opd.py \
    --dataset medquad \
    --hf_teacher_model Qwen/Qwen3.5-9B \
    --student_base_model Qwen/Qwen3.5-0.8B \
    --student_opd_model opd_checkpoints/student_best_model_step_450.pt \
    --eval_samples 100

📦 正在加载内置数据集: medquad
README.md: 100% 233/233 [00:00<00:00, 1.21MB/s]
medDataset_processed.csv: 100% 22.5M/22.5M [00:01<00:00, 13.2MB/s]
Generating train split: 100% 16407/16407 [00:00<00:00, 59341.13 examples/s]
tokenizer_config.json: 100% 16.7k/16.7k [00:00<00:00, 25.2MB/s]
vocab.json: 100% 6.72M/6.72M [00:00<00:00, 137MB/s]
merges.txt: 100% 3.35M/3.35M [00:00<00:00, 119MB/s]
tokenizer.json: 100% 12.8M/12.8M [00:01<00:00, 11.9MB/s]
chat_template.jinja: 100% 7.75k/7.75k [00:00<00:00, 17.5MB/s]
modules.json: 100% 349/349 [00:00<00:00, 1.86MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 744kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 31.8MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 338kB/s]
config.json: 100% 612/612 [00:00<00:00, 3.50MB/s]
model.safetensors: 100% 90.9M/90.9M [00:01<00:00, 69.5MB/s]
Loading weights: 100% 103/103 [00:00<00:00, 6052.47it/s]
tokenizer_config.json: 100% 350/350 [00:00<00:00, 1.56MB/s]
vocab.txt: 100% 232k/232k [00:00<00